# SoICT 2026 paper figures

Hai hình đang dùng trong `paper/soict2026.tex`:

| Label | Nguồn | Sinh bởi |
|---|---|---|
| `fig:inert` (Fig. 1) | `paper/inert_stage.tex` (TikZ, `\input` trực tiếp) | LaTeX → cell bên dưới render standalone để xem |
| `fig:activity` (Fig. 2) | `paper/figs/fig_forced.pdf` | `evaluation/paper_figures.py::fig_forced`, copy sang cell bên dưới |

Chạy notebook từ thư mục `notebooks/` (kernel = `.venv`). Fig 2 cần log trong `output_logs/macsql_drspider/`.

## Fig. 1 — `fig:inert`: cơ chế inert stage (TikZ)

Mã TikZ đúng như file `paper/inert_stage.tex`. Cell này bọc nó vào một document `standalone`, chạy `pdflatex`, rồi hiển thị PNG.

In [ ]:
TIKZ_SRC = r'''% inert_stage.tex — Figure 1
% Included from soict2026.tex via \input{inert_stage}
% Requires: \usetikzlibrary{shapes.geometric, arrows.meta, positioning, fit}
%
% The inert-stage mechanism, on a real logged instance
% (drspider_DB_schema_abbreviation_63, MAC-SQL / gpt-5.6-luna): a fault injected at
% the Schema stage cannot be attributed there, because the Selector emitted no
% decision at all on this database.

\begin{tikzpicture}[
  font=\small,
  stage/.style   = {rectangle, rounded corners=2pt, draw=black!70, thick,
                    minimum width=2.3cm, minimum height=0.95cm,
                    align=center, fill=blue!6},
  faulty/.style  = {stage, fill=orange!22, draw=orange!75!black,
                    dash pattern=on 3pt off 2pt},
  blamed/.style  = {stage, fill=red!16, draw=red!70!black},
  logbox/.style  = {rectangle, draw=black!35, dashed, inner sep=3pt,
                    align=center, font=\scriptsize, fill=black!2},
  note/.style    = {align=left, font=\scriptsize},
  fwd/.style     = {-{Stealth[length=2.2mm]}, thick, black!75}
]

% ---------- pipeline stages ----------
\node[stage]                            (intent)  at (0,0) {\textbf{1. Intent}\\\scriptsize agent};
\node[faulty, right=1.0cm of intent]    (schema)  {\textbf{2. Schema}\\\scriptsize selector};
\node[blamed, right=1.0cm of schema]    (skeleton){\textbf{3. Skeleton}\\\scriptsize agent};
\node[stage,  right=1.0cm of skeleton]  (exec)    {\textbf{4. Execution}\\\scriptsize agent};

\draw[fwd] (intent)   -- (schema);
\draw[fwd] (schema)   -- (skeleton);
\draw[fwd] (skeleton) -- (exec);

% ---------- input, above the row ----------
\node[note, text width=13.6cm, anchor=south west] (q) at ([xshift=-0.4cm, yshift=1.15cm]intent.north west)
  {\textbf{Input $x_1$} (\texttt{world\_1}: 4 tables, 26 columns): ``What are
   the Asian countries which have a population larger than that of any
   country in Africa?''\\[1pt]
   \textcolor{orange!75!black}{\textbf{Perturbation (Schema):}} columns
   abbreviated --- \texttt{population}\,$\to$\,\texttt{popn},
   \texttt{continent}\,$\to$\,\texttt{cont}};

\draw[fwd] ([yshift=-2pt]q.south -| intent.north) -- (intent.north);

% ---------- callouts, below the row ----------
\node[note, text width=3.0cm, anchor=north] (c1) at ([yshift=-0.75cm]schema.south)
  {\textcolor{orange!75!black}{\textbf{fault injected here}} --- but
   $26 \leq 30$ columns and average $6 \leq 6$, so the pruning threshold is not
   met, $x_5 = x_4$, and \emph{no decision is emitted}};

\node[note, text width=3.0cm, anchor=north] (c2) at ([yshift=-0.75cm]skeleton.south)
  {\textcolor{red!70!black}{\textbf{attributed here}} --- the query bounds
   with \texttt{MAX(popn)} over Africa where the gold query uses
   \texttt{MIN}; the abbreviated columns are all resolved correctly};

\draw[black!45, dotted, thick] (schema.south)   -- ([yshift=1pt]c1.north);
\draw[black!45, dotted, thick] (skeleton.south) -- ([yshift=1pt]c2.north);

% ---------- logging matrix, at the bottom ----------
\node[logbox, anchor=north] (l1) at ([yshift=-3.15cm]intent.south)   {$x_1,x_2,x_3$};
\node[logbox, anchor=north] (l2) at ([yshift=-3.15cm]schema.south)   {$x_4,x_5,x_6$\ \ \textcolor{black!55}{($x_5 \equiv x_4$)}};
\node[logbox, anchor=north] (l3) at ([yshift=-3.15cm]skeleton.south) {$x_7,x_8,x_9$};
\node[logbox, anchor=north] (l4) at ([yshift=-3.15cm]exec.south)     {$x_{10}\ldots x_{13}$};

\node[draw=black!45, dashed, rounded corners=2pt, inner sep=5pt,
      fit=(l1)(l4), label={[font=\scriptsize\bfseries, yshift=-1pt]below:%
      Logging Matrix (13 state variables)}] (matrix) {};

\end{tikzpicture}
'''

from pathlib import Path
import subprocess, shutil, tempfile
from IPython.display import Image, display

ROOT = Path.cwd().resolve().parent
OUT = ROOT / 'paper' / 'figs'
OUT.mkdir(exist_ok=True)

DOC = r'''\documentclass[border=4pt]{standalone}
\usepackage{xcolor}
\usepackage{tikz}
\usetikzlibrary{shapes.geometric, arrows.meta, positioning, fit}
\begin{document}
''' + TIKZ_SRC + r'''
\end{document}
'''

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    (td / 'inert_stage_standalone.tex').write_text(DOC)
    subprocess.run(['pdflatex', '-interaction=nonstopmode', 'inert_stage_standalone.tex'],
                   cwd=td, check=True, stdout=subprocess.DEVNULL)
    shutil.copy(td / 'inert_stage_standalone.pdf', OUT / 'fig_inert_standalone.pdf')
    subprocess.run(['pdftoppm', '-png', '-r', '150', '-singlefile',
                    str(td / 'inert_stage_standalone.pdf'), str(td / 'inert')], check=True)
    png = (td / 'inert.png').read_bytes()

print('wrote', OUT / 'fig_inert_standalone.pdf')
display(Image(png))


## Fig. 2 — `fig:activity`: forced-Selector arm (`fig_forced`)

Setup + hàm `fig_forced` copy nguyên từ `evaluation/paper_figures.py` (chỉ đổi `ROOT` cho notebook và `return fig` để hiển thị inline).

In [ ]:
"""
evaluation/paper_figures.py

Every figure in paper/soict2026.tex, regenerated from the logs so that no
figure is a file whose provenance has been forgotten. Writes vector PDFs to
paper/figs/.

    python evaluation/paper_figures.py            # all
    python evaluation/paper_figures.py activity   # one, by name

Palette: Okabe–Ito (colour-blind safe), assigned by entity in fixed order and
never cycled. Backbone identity: luna = blue, Qwen = orange. Stage activity:
inert = grey, active = blue. One axis per panel; no dual axes.
"""
import glob
import json
import math
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path.cwd().resolve().parent  # notebook lives in <repo>/notebooks/
sys.path.insert(0, str(ROOT / "evaluation"))
from cross_system_report import GT_STAGE, attribute_baseline, columns_kept, gold_column_refs  # noqa: E402
from label_validity import classify, VERDICTS  # noqa: E402

OUT = ROOT / "paper" / "figs"
OUT.mkdir(exist_ok=True)

STAGES = ["Intent", "Schema", "Skeleton", "Execution"]
BACKBONES = [("gpt-5.6-luna", "openai-gpt-5.6-luna"),
             ("Qwen2.5-14B", "ollama-qwen2.5-14b")]

# Okabe–Ito
C_BLUE, C_ORANGE, C_GREEN, C_VERM, C_GREY, C_SKY, C_PURPLE = (
    "#0072B2", "#E69F00", "#009E73", "#D55E00", "#8C8C8C", "#56B4E9", "#CC79A7")
C_BB = {"gpt-5.6-luna": C_BLUE, "Qwen2.5-14B": C_ORANGE}
C_ACT = {"inert": C_GREY, "pruned": C_BLUE}
INK, INK2 = "#222222", "#555555"

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.linewidth": 0.6,
    "axes.edgecolor": INK2,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.color": INK2,
    "ytick.color": INK2,
    "axes.labelcolor": INK,
    "text.color": INK,
    "pdf.fonttype": 42,
    "legend.frameon": False,
    "legend.fontsize": 7,
})


# ----------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------
def _read_json_objects(path: Path) -> list:
    """Some counterfactual files hold several concatenated JSON objects."""
    txt = path.read_text(encoding="utf-8")
    dec = json.JSONDecoder()
    i, out = 0, []
    while i < len(txt):
        while i < len(txt) and txt[i] in " \n\r\t":
            i += 1
        if i >= len(txt):
            break
        try:
            d, i = dec.raw_decode(txt, i)
        except json.JSONDecodeError:
            break
        out.append(d)
    return out


def load_drspider(backbone_dir: str) -> list:
    path = ROOT / "output_logs" / "macsql_drspider" / backbone_dir
    out = []
    for f in sorted(glob.glob(str(path / "drspider_*.json"))):
        try:
            out.append(json.loads(Path(f).read_text(encoding="utf-8")))
        except Exception:
            continue
    return out


def load_bird(system: str) -> list:
    path = ROOT / "output_logs" / f"{system}_bird" / "openai-gpt-5.6-luna"
    out = []
    for f in sorted(glob.glob(str(path / "bird_*.json"))):
        try:
            out.append(json.loads(Path(f).read_text(encoding="utf-8")))
        except Exception:
            continue
    return out


def load_counterfactual(arm: str) -> dict:
    path = ROOT / "output_logs" / "counterfactual" / "openai-gpt-5.6-luna" / arm
    m = {}
    for f in glob.glob(str(path / "*.json")):
        for d in _read_json_objects(Path(f)):
            if isinstance(d, dict) and "benchmark_id" in d:
                m[d["benchmark_id"]] = d
    return m


def pruned(d: dict) -> bool:
    return bool((d.get("x5") or {}).get("pruned"))


def wilson(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (c - h, c + h)


def _style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#DDDDDD", lw=0.5, zorder=0)
    ax.set_axisbelow(True)




In [ ]:
def fig_forced() -> None:
    """DB group under the default sweep and the forced-Selector arm.

    Four bars per backbone: instances the default Selector left inert vs
    pruned, each under the default sweep and under the arm that forces the
    Selector to act. (a) EX barely moves; (b) schema recall on failures is
    zero on inert instances only while the Selector is inert. A backbone
    without a forced arm on disk is drawn with its default bars only.
    """
    rows = {}
    for bb, bdir in BACKBONES:
        base = {d["benchmark_id"]: d for d in load_drspider(bdir)
                if d.get("perturbation_group") == "DB"}
        arm = {d["benchmark_id"]: d for d in load_drspider(bdir + "_selector-force")
               if d.get("perturbation_group") == "DB"}
        for act in ("inert", "pruned"):
            ids = [i for i in base if pruned(base[i]) == (act == "pruned")]
            for arm_name, logs in (("default", base), ("forced", arm)):
                sub = [logs[i] for i in ids if i in logs]
                if len(sub) < len(ids):  # arm not run for this backbone
                    continue
                fails = [d for d in sub if not d.get("execution_match")]
                rows[(bb, act, arm_name)] = dict(
                    n=len(sub), ex=sum(bool(d.get("execution_match")) for d in sub),
                    nf=len(fails), hit=sum(attribute_baseline(d) == "Schema" for d in fails))

    fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.3))
    w = 0.19
    order = [("inert", "default"), ("inert", "forced"), ("pruned", "default"), ("pruned", "forced")]
    for ax, key, ylabel, title in (
        (axes[0], "ex", "execution accuracy (%)", "(a) forcing the Selector to act leaves EX unchanged"),
        (axes[1], "rec", "schema recall on failures (%)", "(b) recall leaves zero only where the freed Selector prunes"),
    ):
        for j, (bb, _) in enumerate(BACKBONES):
            for k, (act, arm_name) in enumerate(order):
                r = rows.get((bb, act, arm_name))
                x = j + (k - 1.5) * (w + 0.03)
                if r is None:
                    continue
                num, den = (r["ex"], r["n"]) if key == "ex" else (r["hit"], r["nf"])
                v = 100 * num / den if den else 0
                lo, hi = wilson(num, den)
                ax.bar(x, v, w, color=C_ACT[act], edgecolor="white" if arm_name == "default" else INK,
                       hatch="" if arm_name == "default" else "////", lw=0.6, zorder=3)
                ax.errorbar(x, v, yerr=[[v - 100 * lo], [100 * hi - v]],
                            fmt="none", ecolor=INK, elinewidth=0.7, capsize=1.5, zorder=4)
                ax.text(x, 100 * hi + 1.5, f"{num}/{den}", ha="center", va="bottom",
                        fontsize=5.5, color=INK, rotation=90)
        ax.set_xticks([0, 1])
        ax.set_xticklabels([bb for bb, _ in BACKBONES])
        ax.set_ylabel(ylabel)
        ax.set_ylim(0, 112 if key == "ex" else 80)
        ax.set_title(title, fontsize=7.5, loc="left")
        _style(ax)
    fig.legend(handles=[Patch(color=C_ACT["inert"], label="Selector inert in the default sweep"),
                        Patch(color=C_ACT["pruned"], label="Selector pruned in the default sweep"),
                        Patch(facecolor="white", edgecolor=INK, hatch="////", label="forced to act")],
               ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.08), handlelength=1.4)
    fig.tight_layout()
    fig.savefig(OUT / "fig_forced.pdf", bbox_inches="tight")
    print("wrote", OUT / "fig_forced.pdf")
    return fig


fig = fig_forced()
fig
